In [7]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY not found in .env file")

print("OpenRouter API key loaded successfully.")

OpenRouter API key loaded successfully.


In [8]:
# from langchain_openrouter import ChatOpenRouter

# model = ChatOpenRouter(
#     model="z-ai/glm-5.3-flash",
#     base_url="https://openrouter.ai/api/v1",
#     api_key=OPENROUTER_API_KEY,
#     temperature=0
# )
# model2 = ChatOpenRouter(
#     model="google/gemma-4-31b-it:free",
#     base_url="https://openrouter.ai/api/v1",
#     api_key=OPENROUTER_API_KEY,
#     temperature=0
# )

In [9]:
from langchain_groq import ChatGroq
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
model = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=GROQ_API_KEY,
    temperature=0
)

In [10]:
response = model.invoke(
    "What is MCP?"
)
print(response.content)

**MCP** can stand for many different things depending on the context. Here are a few of the most common meanings:

| Field | Full Form | Brief Description |
|-------|-----------|-------------------|
| **Computing / Hardware** | **Microchip** | Often abbreviated as “MCP” in the context of microcontroller families (e.g., Microchip Technology’s PIC, AVR, SAM, etc.). |
| **Computing / Software** | **Master Control Program** | The central operating system in the 1982 film *Tron* (and its sequels). |
| **IT Certifications** | **Microsoft Certified Professional** | A credential for IT professionals who have passed Microsoft certification exams. |
| **Networking** | **Multi‑Channel Processor** | A device or software that handles multiple communication channels simultaneously. |
| **Chemistry / Biology** | **Methyl‑CpG‑binding Protein** | A protein that recognizes methylated DNA (often abbreviated as “MCP” in genomics). |
| **Chemokines** | **Monocyte Chemoattractant Protein‑1 (MCP‑1)** | A cyt

In [11]:
from mcp.shared.exceptions import McpError

print("MCP ERROR OK")

MCP ERROR OK


In [12]:
import asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent

async def main():

    print("1. Creating client...")

    client = MultiServerMCPClient({
        "mytools": {
            "transport": "http",
            "url": "http://localhost:8000/mcp",
        }
    })

    print("2. Getting tools...")

    tools = await client.get_tools()

    print("3. Tools received:")
    for tool in tools:
        print("-", tool.name)

    print("4. Creating agent...")

    agent = create_agent(
        model,
        tools
    )

    print("5. Calling agent...")

    response = await agent.ainvoke({
        "messages": [
            {
                "role": "user",
                "content": "What is the sum of (3+2)?"
            }
        ]
    })

    print("6. Agent finished!")

    print(response["messages"][-1].content)


await main()

1. Creating client...
2. Getting tools...
3. Tools received:
- add
- multiply
- get_weather
4. Creating agent...
5. Calling agent...
6. Agent finished!
The sum of \(3 + 2\) is **5**.


In [13]:
# async def create_my_agent():
#     client = MultiServerMCPClient(
#         {
#             "mytools":{
#                 "transport": "http",
#                 "url": "http://localhost:8000/mcp",
#             }
#         }
#     )
#     tools = await client.get_tools()

#     agent = create_agent(model,tools)
#     return agent
# async def run_agent(agent,question,agent_name):
#     print(f"{agent_name} started")

#     response = await agent.ainvoke({
#         "messages": [
#             {
#                 "role": "user",
#                 "content": question
#             }
#         ]
#     })

#     answer = response["messages"][-1].content

#     print(f"{agent_name} finished.")

#     return agent_name, answer


In [14]:
async def run_agent(agent, question, agent_name):

    print(f"{agent_name} started...")

    print(f"{agent_name}: calling LLM...")
    
    response = await agent.ainvoke({
        "messages": [
            {
                "role": "user",
                "content": question
            }
        ]
    })

    print(f"{agent_name}: response received!")

    answer = response["messages"][-1].content

    print(f"{agent_name} finished.")

    return agent_name, answer

In [15]:
#Having different agents and fetching the tools dyncmically and running parallely 
import asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent

async def main():

    client = MultiServerMCPClient({
        "mytools": {
            "transport": "http",
            "url": "http://localhost:8000/mcp",
        }
    })

    print("Getting MCP tools...")
    tools = await client.get_tools()

    print("Tools:")
    for tool in tools:
        print("-", tool.name)

    # Three agents sharing the same tool definitions
    agent1 = create_agent(model, tools)
    agent2 = create_agent(model, tools)
    agent3 = create_agent(model, tools)

    results = await asyncio.gather(
    run_agent(
        agent1,
        "Use the add tool to calculate 1 + 2.",
        "Agent 1"
    ),
    run_agent(
        agent2,
        "what is the weather of California?",
        "Agent 2"
    ),
    run_agent(
        agent3,
        "Use the multiplication tool and calculate 5 * 3.",
        "Agent 3"
    ),
    )
    print("\n========== RESULTS ==========")

    for agent_name, answer in results:
        print(f"\n{agent_name}:")
        print(answer)


await main()

Getting MCP tools...
Tools:
- add
- multiply
- get_weather
Agent 1 started...
Agent 1: calling LLM...
Agent 2 started...
Agent 2: calling LLM...
Agent 3 started...
Agent 3: calling LLM...
Agent 1: response received!
Agent 1 finished.
Agent 2: response received!
Agent 2 finished.
Agent 3: response received!
Agent 3 finished.

========== RESULTS ==========

Agent 1:
The result of 1 + 2 is **3**.

Agent 2:
It’s always sunny in California.

Agent 3:
15


In [16]:
#MultiAgent System with A supervisor agent and handling the task delegation dynamically 
client = MultiServerMCPClient({
        "mytools": {
            "transport": "http",
            "url": "http://localhost:8000/mcp",
        }
    })
tools = await client.get_tools()
math_tools = [
    tool for tool in tools
    if tool.name in ["add", "multiply"]
]

weather_tools = [
    tool for tool in tools
    if tool.name == "get_weather"
]

math_agent = create_agent(
    model,
    math_tools,
    system_prompt="""
You are a math specialist.

Your job is to solve mathematical calculations.
Use the available math tools whenever appropriate.

Return a concise answer with the calculation result.
"""
)

weather_agent = create_agent(
    model,
    weather_tools,
    system_prompt="""
You are a weather specialist.

Your job is to answer weather-related questions.
Use the get_weather tool whenever weather information is requested.

Return a concise answer.
"""
)





In [17]:
async def run_math_agent(task: str):
    result = await math_agent.ainvoke({
        "messages": [
            ("human", task)
        ]
    })

    return result["messages"][-1].content

async def run_weather_agent(task:str):
    result =  await weather_agent.ainvoke({
        "mesaages":[
            ("human",task)
        ]
    })

In [18]:
from langchain.tools import tool
@tool
async def delegate_to_math_agent(task: str) -> str:
    """Delegate a mathematical task to the math specialist."""
    
    result = await math_agent.ainvoke({
        "messages": [
            ("human", task)
        ]
    })

    return result["messages"][-1].content

@tool
async def delegate_to_weather_agent(task: str) -> str:
    """Delegate a weather-related task to the weather specialist."""
    
    result = await weather_agent.ainvoke({
        "messages": [
            ("human", task)
        ]
    })

    return result["messages"][-1].content

In [19]:
supervisor = create_agent(
    model,
    [
        delegate_to_math_agent,
        delegate_to_weather_agent
    ],
    system_prompt="""
You are a supervisor responsible ONLY for task delegation and
result synthesis.

You have two specialist agents:

1. Math Agent
   - Handles all mathematical calculations.

2. Weather Agent
   - Handles all weather-related questions.

RULES:

- Do NOT solve mathematical calculations yourself.
- Do NOT answer weather questions yourself.
- ALWAYS delegate the appropriate task to the relevant specialist.
- If the user's request contains multiple independent tasks,
  delegate all of them in parallel when possible.
- After receiving the specialist results, combine them into one
  concise final answer.
"""
)

In [20]:
result = await supervisor.ainvoke({
    "messages": [
        ("human", "Calculate 15 * 7")
    ]
})

print(result["messages"][-1].content)

The result of \(15 \times 7\) is **105**.


In [21]:
result = await supervisor.ainvoke({
    "messages": [
        ("human", "What's the weather in California?")
    ]
})

print(result["messages"][-1].content)

The weather in California is always sunny.


In [22]:
result = await supervisor.ainvoke({
    "messages": [
        ("human", "What's the weather in California? and also tell me the sum of 12*6?")
    ]
})

print(result["messages"][-1].content)


California’s weather is sunny, and 12 × 6 equals 72.


In [23]:
print(result)

{'messages': [HumanMessage(content="What's the weather in California? and also tell me the sum of 12*6?", additional_kwargs={}, response_metadata={}, id='1d24cb06-f0d3-40e2-9744-3057cd302fd6'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We have two tasks: weather in California (weather question) and sum of 12*6 (math). We must delegate both. Use functions.', 'tool_calls': [{'id': 'fc_c496b3ba-cd1d-4031-9050-136dbf496d2d', 'function': {'arguments': '{"task":"What\'s the weather in California?"}', 'name': 'delegate_to_weather_agent'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 284, 'total_tokens': 345, 'completion_time': 0.063844108, 'completion_tokens_details': {'reasoning_tokens': 31}, 'prompt_time': 0.003009123, 'prompt_tokens_details': {'cached_tokens': 256}, 'queue_time': 0.306729751, 'total_time': 0.066853231}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_851252edc2', 'service_tier': 'on_d

In [24]:
result = await supervisor.ainvoke({
    "messages": [
        (
            "human",
            "What's the weather in California and what is 12 * 6?"
        )
    ]
})

print(result["messages"][-1].content)

The weather in California is sunny, and 12 × 6 equals 72.


In [30]:
print(result)

{'messages': [HumanMessage(content="What's the weather in California and what is 12 * 6?", additional_kwargs={}, response_metadata={}, id='0f6e0b7b-6f20-4c6f-a2dd-60dc4c4d1d66'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We have a user request: "What\'s the weather in California and what is 12 * 6?" Two independent tasks: weather question and math calculation. According to rules, we must delegate each to the appropriate specialist. So we need to call delegate_to_weather_agent with task "What\'s the weather in California?" and delegate_to_math_agent with task "12 * 6". Then after receiving results, combine into one concise final answer. We need to use the function calls. We\'ll call both.', 'tool_calls': [{'id': 'fc_d3e21337-6a48-4ba7-a5e3-7909cd3cdcf5', 'function': {'arguments': '{"task":"What\'s the weather in California?"}', 'name': 'delegate_to_weather_agent'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 130, 'prompt_tokens': 2

In [33]:
for i, message in enumerate(result["messages"]):

    print(f"\n========== MESSAGE {i} ==========")
    print("TYPE:", message.type)

    if message.content:
        print("CONTENT:", message.content)

    if hasattr(message, "tool_calls") and message.tool_calls:
        print("\nTOOL CALLS:")

        for call in message.tool_calls:
            print("Tool:", call["name"])
            print("Args:", call["args"])
            print("ID:", call["id"])


========== MESSAGE 0 ==========
TYPE: human
CONTENT: What's the weather in California and what is 12 * 6?

========== MESSAGE 1 ==========
TYPE: ai

TOOL CALLS:
Tool: delegate_to_weather_agent
Args: {'task': "What's the weather in California?"}
ID: fc_d3e21337-6a48-4ba7-a5e3-7909cd3cdcf5

========== MESSAGE 2 ==========
TYPE: tool
CONTENT: It’s always sunny in California.

========== MESSAGE 3 ==========
TYPE: ai

TOOL CALLS:
Tool: delegate_to_math_agent
Args: {'task': '12 * 6'}
ID: fc_297af58c-1432-4ff5-a87a-9ebe3f607b17

========== MESSAGE 4 ==========
TYPE: tool
CONTENT: 72

========== MESSAGE 5 ==========
TYPE: ai
CONTENT: The weather in California is sunny, and 12 × 6 equals 72.


In [40]:
result = await supervisor.ainvoke({
    "messages": [
        (
            "human",
            """
            I need two completely independent tasks:
            1. Tell me the weather in California.
            2. Calculate 12345 * 678.

            These tasks are independent. Delegate BOTH tasks to the
            appropriate specialist agents in the same turn.
            """
        )
    ]
})

In [41]:
for i, message in enumerate(result["messages"]):

    if message.type == "ai" and message.tool_calls:

        print(f"\n========== AI MESSAGE {i} ==========")

        for call in message.tool_calls:
            print("Tool:", call["name"])
            print("Args:", call["args"])
            print("ID:", call["id"])


========== AI MESSAGE 1 ==========
Tool: delegate_to_weather_agent
Args: {'task': 'Tell me the weather in California.'}
ID: fc_8c311d0b-5077-4aa3-988c-dcb83db6a6d5

========== AI MESSAGE 3 ==========
Tool: delegate_to_math_agent
Args: {'task': 'Calculate 12345 * 678'}
ID: fc_201d6b67-e853-4edc-b86f-e0c4d17e23cd


In [42]:
for i, message in enumerate(result["messages"]):
    if message.type == "ai" and message.tool_calls:
        print(f"\nAI MESSAGE {i}")

        for call in message.tool_calls:
            print("  ", call["name"], "→", call["args"])


AI MESSAGE 1
   delegate_to_weather_agent → {'task': 'Tell me the weather in California.'}

AI MESSAGE 3
   delegate_to_math_agent → {'task': 'Calculate 12345 * 678'}


**Let us now get the supervisor delegate the task to both agent into one go and not rely on prompt only**
we will use asyncio.gather() which is more reliable

In [61]:
# Making one structred state for each task
from pydantic import BaseModel
from typing import Literal

class Task(BaseModel):
    agent:Literal["weather","math"]
    task:str
class Plan(BaseModel):
    tasks:list[Task]

planner = model.with_structured_output(Plan)



In [62]:
plan = await planner.ainvoke(
    """
    Break the user's request into independent tasks.

    User request:
    What's the weather in California and what is 12345 * 678?

    Assign each task to the appropriate specialist.
    """
)
print(response)

tasks=[Task(agent='weather', task="What's the weather in California?"), Task(agent='math', task='What is 12345 * 678?')]


In [63]:
agents = {
    "weather":weather_agent,
    "math":math_agent
}
async def execute_task(task):

    agent = agents[task.agent]

    print(f"START → {task.agent}: {task.task}")

    result = await agent.ainvoke({
        "messages": [
            ("human", task.task)
        ]
    })

    answer = result["messages"][-1].content

    print(f"END   → {task.agent}: {answer}")

    return {
        "agent": task.agent,
        "task": task.task,
        "answer": answer
    }

In [64]:
results = await asyncio.gather(
    *(execute_task(task) for task in plan.tasks)
)

START → weather: What's the weather in California?
START → math: What is 12345 * 678?
END   → weather: It’s always sunny in California.
END   → math: 8,369,910


In [65]:
print("\n========== RESULTS ==========")

for result in results:
    print(f"{result['agent']} → {result['answer']}")


========== RESULTS ==========
weather → It’s always sunny in California.
math → 8,369,910


In [66]:
results_text = "\n".join(
    f"{r['agent']}: {r['answer']}"
    for r in results
)

final = await model.ainvoke(
    f"""
    Answer the user's request using the specialist results below.

    Specialist results:
    {results_text}

    Give a concise final answer.
    """
)

print(final.content)

Weather: It’s always sunny in California.  
Math result: 8,369,910.


In [67]:
import time

async def execute_task(task):

    agent = agents[task.agent]

    start = time.perf_counter()

    print(
        f"[{start:.2f}] START → "
        f"{task.agent}"
    )

    result = await agent.ainvoke({
        "messages": [
            ("human", task.task)
        ]
    })

    end = time.perf_counter()

    answer = result["messages"][-1].content

    print(
        f"[{end:.2f}] END   → "
        f"{task.agent} "
        f"({end-start:.2f}s)"
    )

    return {
        "agent": task.agent,
        "answer": answer
    }

In [68]:
start = time.perf_counter()

results = await asyncio.gather(
    *(execute_task(task) for task in plan.tasks)
)

print(
    f"\nTOTAL: "
    f"{time.perf_counter() - start:.2f}s"
)

[286119.93] START → weather
[286119.94] START → math
[286122.38] END   → weather (2.45s)
[286122.59] END   → math (2.65s)

TOTAL: 2.66s
